# 02 — Document Intelligence smoke test

Uploads a sample PDF, runs `prebuilt-layout`, prints the first text chunk, any tables found, and extracts embedded images — each image is uploaded to Blob and described by GPT-4o vision so it can be retrieved by the RAG layer and shown to users.

**Place** any PDF at `notebooks/sample.pdf` before running.

In [1]:
import sys, os, pathlib
sys.path.insert(0, os.path.abspath('..'))
from src.blob_client import BlobService
from src.doc_intelligence import DocIntelService
from src.openai_client import OpenAIService

PDF_PATH = pathlib.Path('Multi_Agent_Research_System_Architecture.pdf')
assert PDF_PATH.exists(), 'Drop a sample.pdf next to this notebook'

blob = BlobService()
blob.ensure_container()
name = 'docmind-test/sample.pdf'
pdf_bytes = PDF_PATH.read_bytes()
blob.upload(name, pdf_bytes, content_type='application/pdf')
url = blob.get_url(name)
print('PDF URL:', url[:80], '…')

INFO: Incomplete environment configuration for EnvironmentCredential. These variables are set: AZURE_TENANT_ID
INFO: ManagedIdentityCredential will use IMDS
INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input?restype=REDACTED&sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'x-ms-version': 'REDACTED'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '1aec1cc7-4a51-11f1-8d8d-bcd22c162bea'
No body was attached to the request
INFO: Response status: 403
Response headers:
    'Content-Length': '246'
    'Content-Type': 'application/xml'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '61590969-901e-0088-6f5d-deb0f4000000'
    'x-ms-client-request-id': '1aec1cc7-4a51-11f1-8d8d-bcd22c162bea'
    'x-ms-version': 'REDAC

PDF URL: https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/sample.p …


In [2]:
doc_intel = DocIntelService()
result = doc_intel.extract_pdf(url)
print('pages:', result['pages'])
print('text chunks:', len(result['text_chunks']))
print('tables:', len(result['tables']))
print('--- first 600 chars of page 1 ---')
print(result['text_chunks'][0]['content'][:600] if result['text_chunks'] else '(empty)')

INFO: Document Intelligence analysing https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/sample.p
INFO: Request URL: 'https://azdocumentai.cognitiveservices.azure.com//documentintelligence/documentModels/prebuilt-layout:analyze?api-version=2024-11-30&outputContentFormat=REDACTED'
Request method: 'POST'
Request headers:
    'content-type': 'application/json'
    'Content-Length': '237'
    'Accept': 'application/json'
    'x-ms-client-request-id': '200153c5-4a51-11f1-ae06-bcd22c162bea'
    'User-Agent': 'azsdk-python-ai-documentintelligence/1.0.2 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'Ocp-Apim-Subscription-Key': 'REDACTED'
A body is sent with the request
INFO: Response status: 202
Response headers:
    'Content-Length': '0'
    'Operation-Location': 'REDACTED'
    'x-envoy-upstream-service-time': 'REDACTED'
    'apim-request-id': 'REDACTED'
    'Strict-Transport-Security': 'REDACTED'
    'x-content-type-options': 'REDACTED'
    'x-ms-region': 'REDACTED'
   

pages: 27
text chunks: 26
tables: 24
--- first 600 chars of page 1 ---
Multi-Agent Research System
Architecture Design & Technical Documentation
Author
Kumar Nishant - Senior Al Engineer
Team
MLEDSI
Manager
Krutika Kansara
Date
April 20, 2026
Version
1.2
Status
Draft - For Review


In [3]:
if result['tables']:
    print(result['tables'][0]['content'][:1000])
else:
    print('No tables detected.')

| Author | Kumar Nishant - Senior Al Engineer |
| --- | --- |
| Team | MLEDSI |
| Manager | Krutika Kansara |
| Date | April 20, 2026 |
| Version | 1.2 |
| Status | Draft - For Review |


In [4]:
result.keys()

dict_keys(['pages', 'text_chunks', 'tables', 'figures'])

## Extract embedded images and verbalize them via GPT-4o vision

Document Intelligence does not return raw image bytes, so we use PyMuPDF to pull the embedded images out of the PDF, upload each to Blob Storage, and ask GPT-4o vision to describe it. The text descriptions are what RAG indexes; the `image_url` is what the UI renders so users see the picture when they ask about it.

In [5]:
openai = OpenAIService()
images = doc_intel.extract_images(
    pdf_bytes,
    doc_id='docmind-test',
    blob=blob,
    openai=openai,
    figures=result['figures'],
)
print(f'extracted {len(images)} images')
for img in images[:5]:
    print(f"- page {img['page']}  {img['ext']}  {img['size_bytes']} bytes  -> {img['blob_name']}")

INFO: Request URL: 'https://pdgrag.blob.core.windows.net/sgp-dload05s-ra-input/docmind-test/figures/page15_fig0_0.png?sp=REDACTED&st=REDACTED&se=REDACTED&spr=REDACTED&sv=REDACTED&sr=REDACTED&sig=REDACTED'
Request method: 'PUT'
Request headers:
    'Content-Length': '188895'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-blob-content-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.14.3 (Windows-11-10.0.26200-SP0)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '370fb895-4a51-11f1-a7cb-bcd22c162bea'
A body is sent with the request
INFO: Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Thu, 07 May 2026 20:13:31 GMT'
    'ETag': '"0x8DEAC751BF98AA1"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': '61595b10-901e-0088-715d-deb0f4000000'
    '

extracted 1 images
- page 15  png  188895 bytes  -> docmind-test/figures/page15_fig0_0.png


In [6]:
from IPython.display import Image, Markdown, display

for img in images[:5]:
    display(Markdown(f"### Page {img['page']} — `{img['blob_name']}`"))
    try:
        display(Image(url=img['image_url']))
    except Exception as e:
        print('(could not render inline:', e, ')')
    display(Markdown(f"**Vision description:**\n\n{img['description']}"))

### Page 15 — `docmind-test/figures/page15_fig0_0.png`

**Vision description:**

This image is a flowchart depicting a process for handling user queries through a structured task management system. Below is a detailed description of the components and their relationships:

1. **User Query**: The process begins with a user query.

2. **Planner Node**: Receives the user query and proceeds to analyze intent and define goals.

3. **Analyze Intent + Define Goal**: This step involves understanding the user's intent and setting objectives.

4. **Research Intent Classification**: Classifies the intent into categories such as blog, comparative, deep research, or summary.

5. **Structured Plan Creation**: Develops a structured plan based on the classified intent.

6. **Generate Todos with status=pending**: Creates tasks with a pending status.

7. **Task Manager**: Manages the tasks and feeds into the Todo Selector.

8. **Todo Selector**: Selects pending tasks for processing.

9. **Delegation Layer**: Distributes tasks to various sub-agents.

10. **Sub-Agents (1, 2, N)**: Each sub-agent receives tasks and provides findings and sources.

11. **Aggregator**: Collects findings and sources from sub-agents, then merges, deduplicates, and structures the information.

12. **Context Synthesizer**: Synthesizes the context from the aggregated data.

13. **Todo Tracker**: Marks tasks as completed and stores data.

14. **Todo Checker**: Checks if all tasks are completed. If not, it loops back to the Todo Selector. If all tasks are completed, the process ends.

15. **Research Gap Detected**: If a research gap is detected, it loops back to the Task Manager for further processing.

The flowchart illustrates a cyclical process with feedback loops to ensure all tasks are completed and any research gaps are addressed.